# End-to-end trainig

In [ ]:
LOAD_MODEL = None

In [ ]:
from datetime import datetime
from pathlib import Path

import torch

N_SRC = 10
BATCH_SIZE = 512
N_LCS = 30_000_000
MAX_VAL_SIZE = 128 << 10
PLOT_PARTITION_FRACTION = 1e-3

DP_ROOT = "../../data/dp1"
LSDB_WORKERS = 8
# DEVICE = "cpu"
# DEVICE = "cuda"
DEVICE = torch.device("mps")
# DEVICE = "mps"

PLOT_MAGS = [18, 21, 25]

NOW = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path("./runs") / NOW
PLOT_DIR = RUN_DIR / "plots"

from uncle_val.pipelines.splits import dp1_config, dp2_config
from uncle_val.pipelines import ComputeConfig, TrainingConfig

survey_config = dp1_config(catalog_root=DP_ROOT, n_src=N_SRC, img="diff")
compute_config = ComputeConfig(n_workers=LSDB_WORKERS, device=DEVICE)
training_config = TrainingConfig(
    compute_config=compute_config,
    n_lcs=N_LCS,
    train_batch_size=BATCH_SIZE,
    max_val_size=MAX_VAL_SIZE,
    snapshot_factor=2.0,
    val_batch_size=16 << 10,
    lr=1e-4,
    start_tfboard=True,
)

In [ ]:
import warnings

import dask

dask.config.set(
    {
        "dataframe.convert-string": False,
        "distributed.nanny.environ.ARROW_DEFAULT_MEMORY_POOL": "jemalloc",
    }
)

warnings.filterwarnings(
    "ignore",
    message="Sending large graph of size.*",
    category=UserWarning,
)

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split=None,
    survey_config=survey_config,
    non_extended_only=False,
    model_path=None,
    n_samples=5,
    object_mags=[18, 21, 25],
    compute_config=compute_config,
    subsample_partitions=PLOT_PARTITION_FRACTION,
    output_dir=PLOT_DIR / "data",
)

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir=runs

In [ ]:
from uncle_val.learning.losses import (
    minus_ln_chi2_prob_loss,
    kl_divergence_whiten_loss,
    epps_pulley_whiten_loss,
)
from uncle_val.learning.models import MLPMagErrModel, MLPModel
from uncle_val.pipelines import train_on_rubin_dp

model = LOAD_MODEL or MLPModel(
    input_names=[
        "x",
        "err",
        "psfFlux",
        "extendedness",
        "skyBg",
        "seeing",
        "expTime",
        "detector_rho",
        "detector_cos_phi",
        "detector_sin_phi",
    ]
    + [f"is_{band}_band" for band in survey_config.bands],
    outputs_s=False,
)

model_path, model_columns = train_on_rubin_dp(
    model=model,
    loss_fn=epps_pulley_whiten_loss(lmbd=2.0, soft=20.0, kind="accum"),
    val_losses={
        "Total Soften KL": kl_divergence_whiten_loss(soft=20.0, kind="accum", lmbd=None),
        "Total Soften -ln(p_chi2)": minus_ln_chi2_prob_loss(soft=20.0, kind="accum", lmbd=None),
    },
    output_dir=RUN_DIR,
    survey_config=survey_config,
    training_config=training_config,
    # FILTERS
    # bands=survey_config.bands,
    # pre_filter_partition=lambda df: df.query("r_psfMag < 22.0 and r_extendedness <= 0.5"),
)

In [ ]:
print(model_path)
print(model_columns)

In [ ]:
from uncle_val.pipelines import run_rubin_dp_feature_importance

fig = run_rubin_dp_feature_importance(
    model_path=model_path,
    model_columns=model_columns,
    bands=("r",),
    pre_filter_partition=lambda df: df.query("19.5 <= r_psfMag < 20.5 and r_extendedness <= 0.5"),
    survey_config=survey_config,
    compute_config=compute_config,
)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_DIR / "feature_importance.pdf")

### Train metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="train",
    survey_config=survey_config,
    non_extended_only=False,
    model_path=model_path,
    model_columns=model_columns,
    n_samples=5,
    object_mags=[18, 21, 25],
    compute_config=compute_config,
    subsample_partitions=PLOT_PARTITION_FRACTION,
    output_dir=PLOT_DIR / "model_train",
)

### Validation metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="val",
    survey_config=survey_config,
    non_extended_only=False,
    model_path=model_path,
    model_columns=model_columns,
    n_samples=5,
    object_mags=[18, 21, 25],
    compute_config=compute_config,
    subsample_partitions=PLOT_PARTITION_FRACTION,
    output_dir=PLOT_DIR / "model_validation",
)

### Test metrics

In [ ]:
from uncle_val.pipelines import make_plots

make_plots(
    split="test",
    survey_config=survey_config,
    non_extended_only=False,
    model_path=model_path,
    model_columns=model_columns,
    n_samples=5,
    object_mags=[18, 21, 25],
    compute_config=compute_config,
    subsample_partitions=10 * PLOT_PARTITION_FRACTION,
    output_dir=PLOT_DIR / "model_test",
)